In [1]:
import pandas as pd
import numpy as np
import os
import pickle
from scipy.signal import butter, filtfilt #signal

print("Imported")

Imported


In [2]:
from utils import format_signal, format_flow, format_sleep_profile

In [3]:
d=input("Directory Path")
subjects=[s for s in os.listdir(d)
    if os.path.isdir(os.path.join(d, s))]
l=len(subjects)
print(f"{l} Subject Folders : {subjects}")


Directory Path Data


5 Subject Folders : ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']


In [4]:
#downsampling
#nasal airflow 32Hz, thoracic 32Hz, spo2 4HZ
def resample_signal(flow_df, thorac_df, spo2_df, target_fs=4):
    period=f"{int(1000 / target_fs)}ms"
    ts=flow_df.index[0] #start
    te=flow_df.index[-1] #end
    New_index = pd.date_range(start=ts, end=te, freq=period)
    print(f"Original Flow samples : {len(flow_df):,}")
    print(f"New resampled samples  : {len(New_index):,}")
    
    def resamp(df):
        combined = df.reindex(df.index.union(New_index))
        combined = combined.interpolate(method="time")
        return combined.reindex(New_index)
    flow_r   = resamp(flow_df)
    thorac_r = resamp(thorac_df)
    spo2_r   = resamp(spo2_df)
    return flow_r, thorac_r, spo2_r, New_index, target_fs
print("resample_signals")

resample_signals


In [5]:
def create_windows(flow_arr, thorac_arr, spo2_arr, 
                   timestamps, fs, window_sec=30, overlap=0.5):
    win_size  = int(window_sec * fs)
    step_size = int(window_sec * overlap * fs)
    print(f"Window size  : {win_size} samples = {window_sec} sec")
    print(f"Step size    : {step_size} samples = {window_sec * overlap} sec")
    windows=[]
    i=0
    t=len(flow_arr)
    while i+win_size<=t:
        win_flow=flow_arr[i:i+win_size]
        win_thorac=thorac_arr[i:i+win_size]
        win_spo2=spo2_arr[i: i+win_size]
        win_start = timestamps[i]
        win_end   = timestamps[i + win_size - 1]
        def get_stats(arr):
            return [
                arr.mean(),
                arr.std(),
                arr.min(),
                arr.max(),
                np.percentile(arr, 25),
                np.percentile(arr, 75)
            ]
        # combined stats from3 signals in one row
        features = get_stats(win_flow) + get_stats(win_thorac) + get_stats(win_spo2)
        
        #window info
        windows.append({
            "features" : features,
            "win_start": win_start,
            "win_end"  : win_end
        })
        
        #next window
        i += step_size
    print(f"Total windows created: {len(windows)}")
    return windows

print("create_windows")

create_windows


In [6]:
def filter(signal, fs,low=0.17, high=0.4, order=4):
    nyq=fs/2.0
    low_n=low/nyq
    high_n=high/nyq
    b, a = butter(order, [low_n, high_n], btype="band")
    filtered = filtfilt(b, a, signal)   # ✅ add this line!
    return filtered

print("bandpass_filter")

bandpass_filter


In [7]:
def get_window_label(win_start, win_end, events, threshold=0.5):
    
    window_dur   = (win_end - win_start).total_seconds()
    best_label   = "Normal"
    best_overlap = 0.0

    for ev in events:
        overlap_start = max(win_start, ev["start"])
        overlap_end   = min(win_end,   ev["end"])
        overlap_secs  = max(0.0, (overlap_end - overlap_start).total_seconds())
        overlap_frac  = overlap_secs / window_dur

        if overlap_frac > best_overlap:
            best_overlap = overlap_frac
            best_label   = ev["label"]

    if best_overlap > threshold:
        return best_label
    
    return "Normal"

print("get_window_label")

get_window_label


In [8]:
def process_subject(folder):
    
    subject = os.path.basename(folder)
    
    flow_df   = format_signal(f"{folder}/nasal_airflow.txt")
    thorac_df = format_signal(f"{folder}/thoracic_movement.txt")
    spo2_df   = format_signal(f"{folder}/spo2.txt")
    events    = format_flow(f"{folder}/flow_events.txt")
    sleep_df  = format_sleep_profile(f"{folder}/sleep_profile.txt")
    
    flow_r, thorac_r, spo2_r, New_index, fs = resample_signal(flow_df, thorac_df, spo2_df)
    
    flow_f= filter(flow_r["value"].values,fs)
    thorac_f= filter(thorac_r["value"].values,fs)
    spo2_f= filter(spo2_r["value"].values,fs)
    
    windows = create_windows(flow_f, thorac_f, spo2_f, New_index, fs)
    
    rows = []
    for w in windows:
        label = get_window_label(w["win_start"], w["win_end"], events)
        row   = [subject] + w["features"] + [label, w["win_start"], w["win_end"]]
        rows.append(row)
    
    stat_names  = ["mean", "std", "min", "max", "p25", "p75"]
    signal_names = ["flow", "thorac", "spo2"]
    feat_cols   = [f"{s}_{st}" for s in signal_names for st in stat_names]
    all_cols    = ["subject"] + feat_cols + ["label", "win_start", "win_end"]
    
    df = pd.DataFrame(rows, columns=all_cols)
    print(f" {subject} — {len(df)} windows — {df['label'].value_counts().to_dict()}")
    return df

print("process_subject")

process_subject


In [ ]:
in_dir  = input("Enter Data directory path (e.g. Data): ")
out_dir = input("Enter output directory (e.g. Dataset): ")
os.makedirs(out_dir, exist_ok=True)

subjects = sorted([
    d for d in os.listdir(in_dir)
    if os.path.isdir(os.path.join(in_dir, d))
])

print(f"Found subjects: {subjects}")

all_dfs = []

for p in subjects:
    folder = os.path.join(in_dir, p)
    df     = process_subject(folder)
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)

csv_path = os.path.join(out_dir, "breathing_dataset.csv")
pkl_path = os.path.join(out_dir, "breathing_dataset.pkl")

combined.to_csv(csv_path, index=False)
with open(pkl_path, "wb") as f:
    pickle.dump(combined, f)

print(f"\n✅ Dataset saved → {csv_path}")
print(f"✅ Dataset saved → {pkl_path}")
print(f"\nTotal windows : {len(combined)}")
print(f"\nLabel distribution:\n{combined['label'].value_counts().to_string()}")

Enter Data directory path (e.g. Data):  Data
Enter output directory (e.g. Dataset):  Dataset


Found subjects: ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']
